> **제11회 기출변형 문제** — 교재 533~559쪽
> 문제 533~535쪽 / 정답·해설 536~559쪽
>
> 노트북과 같은 위치에 데이터 파일(csv)을 두고 위에서부터 차례대로 실행하세요.

## 문제 1. (10점)

제공된 데이터(hypertension_risk.csv)는 건강검진에서 수집한 생활습관 및 검사 수치 데이터이다. 이 데이터를 id 기준으로 train(1~700)과 test(701~850)로 나누어 로지스틱 회귀분석을 수행하고, 제시된 소문항에 답하시오.

**제공 데이터 (hypertension_risk.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 검진자 번호 |
| age | 나이 |
| smoking | 흡연 여부 (no/yes) |
| exercise_days | 주간 운동 일수 |
| cholesterol | 콜레스테롤 수치 |
| bmi | 체질량지수 |
| hypertension_risk | (종속변수) 고혈압 위험군 여부 — 0: 저위험, 1: 고위험 |

**문제 1-1** train 데이터에서 id를 제외한 모든 변수를 활용하여 로지스틱 회귀분석을 수행하시오. smoking이 'no'인 사람 대비 'yes'인 사람의 오즈비(Odds Ratio)를 구하시오. (반올림하여 소수 둘째 자리까지 작성)

**문제 1-2** 위 모델로 test 데이터의 고혈압 위험 확률을 예측했을 때, 확률이 가장 높은 사람의 cholesterol 값을 구하시오. (정수로 작성)

**문제 1-3** 위 모델로 test 데이터를 예측한 후, 확률이 0.2 초과인 사람을 고위험군으로 분류했을 때 민감도(Sensitivity)를 구하시오. (반올림하여 소수 둘째 자리까지 작성)

### 문제 풀이 방향

이 문제에서 새롭게 다루는 부분은 문제 1-2입니다. 지금까지는 모델의 예측 결과(확률, 클래스) 자체를 답으로 구했지만, 이번에는 "확률이 가장 높은 사람이 누구인지 찾은 뒤, 그 사람의 다른 원본 데이터 값(cholesterol)을 조회"해야 합니다. 즉 예측 결과와 원본 데이터를 다시 연결하는 작업이 필요합니다. 이를 위해서는 예측 확률이 원본 test 데이터의 몇 번째 행과 대응하는지 인덱스를 정확히 맞추는 것이 중요합니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [1]:
import pandas as pd

df = pd.read_csv('hypertension_risk.csv')
print(df.shape)
df.head()

(850, 7)


,id,age,smoking,exercise_days,cholesterol,bmi,hypertension_risk
0,1,66,no,1,203.9,26.7,1
1,2,31,no,1,196.4,27.9,1
2,3,65,no,2,213.5,27.8,0
3,4,38,yes,3,152.7,19.1,0
4,5,50,yes,6,182.4,27.7,0


850명의 검진자 데이터입니다. smoking은 문자열('no'/'yes')로 되어 있으므로 회귀분석 전에 숫자로 변환해야 합니다.

**셀 2) 범주형 변수 변환 및 train/test 분할**

In [2]:
df['smoking'] = df['smoking'].map({'no': 0, 'yes': 1})

train = df[df['id'] <= 700].reset_index(drop=True)
test = df[df['id'] > 700].reset_index(drop=True)

print("train:", train.shape, "test:", test.shape)

train: (700, 7) test: (150, 7)


`.map({'no': 0, 'yes': 1})`로 no를 0, yes를 1로 변환했습니다. 이 매핑 방향이 문제 1-1의 "no 대비 yes" 오즈비 해석과 일치합니다. `reset_index(drop=True)`는 id 조건으로 걸러낸 뒤 흐트러진 인덱스를 0부터 다시 순서대로 매겨주는 역할로, 이번 문제에서는 특히 중요합니다. 뒤에서 test 데이터의 특정 행을 다시 조회해야 하는데, 인덱스가 깔끔하게 0부터 정렬되어 있어야 예측 결과(pred_prob)의 인덱스와 test 데이터프레임의 인덱스가 정확히 대응되기 때문입니다.

**셀 3) train 데이터로 로지스틱 회귀모형 적합**

In [3]:
import statsmodels.api as sm

X_train = train.drop(columns=['id', 'hypertension_risk'])
y_train = train['hypertension_risk']
X_train = sm.add_constant(X_train)

model = sm.Logit(y_train, X_train).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.597373
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:      hypertension_risk   No. Observations:                  700
Model:                          Logit   Df Residuals:                      694
Method:                           MLE   Df Model:                            5
Date:                Mon, 20 Jul 2026   Pseudo R-squ.:                  0.1323
Time:                        22:34:20   Log-Likelihood:                -418.16
converged:                       True   LL-Null:                       -481.89
Covariance Type:            nonrobust   LLR p-value:                 8.199e-26
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -6.8775      0.864     -7.960      0.000      -8.571      -5.184
age               0.

다섯 변수 모두 p-value가 0.05보다 작아 통계적으로 유의합니다.

**셀 4) 오즈비 계산 (문제 1-1 정답)**

In [4]:
import numpy as np

odds = np.exp(model.params['smoking'])
print("1-1 정답:", round(odds, 2))

1-1 정답: 2.64


smoking의 계수(0.9718)에 지수를 취해 오즈비를 구했습니다. 비흡연자 대비 흡연자의 고혈압 위험 오즈가 약 2.64배 높다는 뜻입니다.

**셀 5) test 데이터의 확률 예측**

In [5]:
X_test = test.drop(columns=['id', 'hypertension_risk'])
X_test = sm.add_constant(X_test, has_constant='add')

pred_prob = model.predict(X_test)
print(pred_prob.head())

0    0.514742
1    0.698137
2    0.349665
3    0.792819
4    0.128666
dtype: float64


`pred_prob`는 test 데이터의 각 행(0번, 1번, 2번, ...)에 대응하는 확률값입니다. 이 인덱스가 test 데이터프레임의 인덱스와 정확히 같은 순서로 대응된다는 점이 다음 셀의 핵심입니다.


**셀 6) 확률이 가장 높은 사람의 cholesterol 조회 (문제 1-2 정답)**

In [6]:
max_idx = pred_prob.idxmax()
print("확률 최댓값 인덱스:", max_idx, "/ 확률:", pred_prob[max_idx])

answer_1_2 = test.loc[max_idx, 'cholesterol']
print("1-2 정답:", answer_1_2)

확률 최댓값 인덱스: 17 / 확률: 0.906295065920243
1-2 정답: 255.0


`pred_prob.idxmax()`는 예측 확률 중 가장 큰 값의 인덱스(몇 번째 행인지)를 돌려줍니다. 이 인덱스는 예측에 사용한 X_test와 같은 순서로 만들어졌으므로, 원본 test 데이터프레임에서도 똑같은 인덱스(17번)가 같은 사람을 가리킵니다. `test.loc[max_idx, 'cholesterol']`로 그 사람의 원본 데이터에서 cholesterol 값을 바로 조회할 수 있습니다. 여기서 cholesterol은 모델의 예측값이 아니라, 모델을 학습시킬 때 이미 사용했던 원본 입력값이라는 점에 주의해야 합니다. 즉 "모델이 새로 계산해낸 값"이 아니라 "이미 갖고 있던 데이터를 다시 찾아보는 것"입니다.

> 💡 예측 결과(Series나 배열)와 원본 데이터프레임은 서로 다른 객체이지만, 같은 순서로 만들어졌다면 인덱스가 서로 대응됩니다. `train_test_split`을 사용하지 않고 조건으로 직접 나눈 데이터라면, 이렇게 `reset_index(drop=True)`로 인덱스를 가지런히 정리해두는 것이 이런 유형의 문제를 풀 때 특히 중요합니다.

**셀 7) 민감도 계산 (문제 1-3 정답)**

In [7]:
pred_class = (pred_prob > 0.2).astype(int)
y_test = test['hypertension_risk']

TP = ((pred_class == 1) & (y_test == 1)).sum()
FN = ((pred_class == 0) & (y_test == 1)).sum()
sensitivity = TP / (TP + FN)

print("TP:", TP, "FN:", FN)
print("1-3 정답:", round(sensitivity, 2))

TP: 63 FN: 5
1-3 정답: 0.93


민감도는 실제 고위험군(TP+FN) 중에서 모델이 제대로 잡아낸 비율입니다. 임계값을 0.2로 낮게 설정했기 때문에, 조금이라도 위험 신호가 있으면 고위험군으로 분류해 실제 고위험군의 93%를 잡아낸 것으로 나타났습니다.

### 전체 코드

In [8]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv('hypertension_risk.csv')
df['smoking'] = df['smoking'].map({'no': 0, 'yes': 1})

train = df[df['id'] <= 700].reset_index(drop=True)
test = df[df['id'] > 700].reset_index(drop=True)

X_train = sm.add_constant(train.drop(columns=['id', 'hypertension_risk']))
y_train = train['hypertension_risk']
model = sm.Logit(y_train, X_train).fit()

# 1-1: 오즈비
odds = round(np.exp(model.params['smoking']), 2)
print("1-1 정답:", odds)

# test 예측
X_test = sm.add_constant(test.drop(columns=['id', 'hypertension_risk']), has_constant='add')
pred_prob = model.predict(X_test)

# 1-2: 확률 최댓값인 사람의 cholesterol
max_idx = pred_prob.idxmax()
print("1-2 정답:", test.loc[max_idx, 'cholesterol'])

# 1-3: 민감도
pred_class = (pred_prob > 0.2).astype(int)
y_test = test['hypertension_risk']
TP = ((pred_class == 1) & (y_test == 1)).sum()
FN = ((pred_class == 0) & (y_test == 1)).sum()
print("1-3 정답:", round(TP / (TP + FN), 2))

Optimization terminated successfully.
         Current function value: 0.597373
         Iterations 5
1-1 정답: 2.64
1-2 정답: 255.0
1-3 정답: 0.93


### 정리

이 문제의 핵심은 예측 확률(모델의 출력)과 원본 데이터(모델의 입력이자 참고 정보)를 인덱스로 정확히 연결하는 것이었습니다. `idxmax()`로 위치를 찾고, 그 위치를 원본 데이터프레임에 그대로 적용해 다른 컬럼 값을 조회하는 흐름은, 1유형에서 다뤘던 "groupby + idxmax로 조건에 맞는 행을 찾은 뒤 다른 컬럼을 조회하는" 패턴과 본질적으로 같은 원리입니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

"확률이 가장 높은 사람"이 아니라 "가장 낮은 사람"을 찾으라는 변형이 나올 수 있습니다.

**변형 예시: 확률이 가장 낮은 사람의 age를 구하라면**

In [9]:
min_idx = pred_prob.idxmin()
print("확률 최솟값 인덱스:", min_idx, "/ 확률:", pred_prob[min_idx])

answer_min = test.loc[min_idx, 'age']
print("확률 최저자의 age:", answer_min)

확률 최솟값 인덱스: 44 / 확률: 0.0900093547604877
확률 최저자의 age: 32


`idxmax()`가 `idxmin()`으로 바뀌기만 하면, 이번엔 고혈압 위험이 가장 낮다고 예측된 사람의 정보를 조회하게 됩니다. 나이가 32세로 비교적 젊다는 것에서, 이 모델이 나이를 위험도 판단에 실제로 반영하고 있다는 것을 짐작할 수 있습니다.

---

## 문제 2. (10점)

제공된 데이터(ad_campaign_sales.csv)는 온라인 광고 캠페인 활동에 따른 일별 매출 데이터이다. 방문자 수, 광고비, 이메일 발송 수, 소셜미디어 반응 수 등에 따른 매출액(sales)에 관한 자료이다. 이 데이터를 이용하여 통계 분석을 수행하고, 제시된 소문항에 답하시오.

**제공 데이터 (ad_campaign_sales.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 일자 번호 |
| visit_count | 방문자 수 |
| ad_spend | 광고비(만원) |
| email_sent | 이메일 발송 수 |
| social_engagement | 소셜미디어 반응 수(좋아요, 댓글 등) |
| sales | (종속변수) 일별 매출액(원) |

**문제 2-1** 일별 매출 평균이 600,000원과 같은지 단일표본 t검정(t-test)을 수행하고, p-value를 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-2** 매출액(sales)과 다른 모든 변수들 간의 피어슨 상관계수를 각각 구하고, 그 중 상관관계가 가장 강한(절댓값이 가장 큰) 상관계수 값을 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-3** 매출액(sales)을 종속변수로, 나머지 모든 변수를 독립변수로 하여 회귀분석을 수행하시오. 절편을 제외한 변수 중에서 p-value가 가장 작은 변수(통계적으로 가장 유의한 변수)의 회귀계수를 구하시오. (반올림하여 소수 셋째 자리까지 작성)

### 문제 풀이 방향

이 문제는 하나의 데이터셋으로 세 가지 서로 다른 통계 기법(단일표본 t검정, 상관분석, 회귀분석)을 순서대로 수행합니다. 지금까지는 한 문제에서 한 가지 통계 기법만 다뤘지만, 이번에는 소문항마다 완전히 다른 도구(`ttest_1samp`, `corrwith`, `sm.OLS`)를 사용해야 합니다. 세 기법 모두 "매출액"이라는 같은 변수를 중심에 두고 있지만, 묻는 질문의 성격(평균이 특정 값과 같은가 / 어떤 변수와 가장 관련이 깊은가 / 어떤 변수가 매출을 가장 잘 설명하는가)이 서로 다르다는 것을 염두에 두고 접근하겠습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [10]:
import pandas as pd

df = pd.read_csv('ad_campaign_sales.csv')
print(df.shape)
df.head()

(240, 6)


,id,visit_count,ad_spend,email_sent,social_engagement,sales
0,1,470,371.6,352,1930,649951.0
1,2,120,166.7,57,214,344611.0
2,3,261,498.8,1356,985,767784.0
3,4,576,51.2,4869,1680,550958.0
4,5,513,237.0,4794,2299,633839.0


240일간의 광고 캠페인 및 매출 데이터입니다.

**셀 2) 단일표본 t검정 수행**

In [11]:
from scipy.stats import ttest_1samp

t_stat, p = ttest_1samp(df['sales'], 600000)
print("t통계량:", t_stat)
print("p-value:", p)

t통계량: 4.152227853883567
p-value: 4.5837744716051916e-05


`ttest_1samp(데이터, 비교값)`은 데이터의 평균이 특정 값(여기서는 600,000원)과 통계적으로 같다고 볼 수 있는지를 검정하는 단일표본 t검정 함수입니다. 지금까지 다룬 대응표본 t검정(`ttest_rel`)이 "같은 대상을 두 번 측정한 차이"를 검정했다면, 단일표본 t검정은 "하나의 데이터 집합의 평균이 특정 기준값과 같은가"를 검정한다는 점이 다릅니다. 귀무가설은 "매출 평균이 600,000원이다"이고, 대립가설은 "매출 평균이 600,000원이 아니다"입니다.

**셀 3) p-value 확인 (문제 2-1 정답)**

In [12]:
answer_2_1 = round(p, 3)
print("2-1 정답:", answer_2_1)

2-1 정답: 0.0


p-value가 약 0.0000458로 유의수준 0.05보다 훨씬 작으므로, "매출 평균이 600,000원이다"라는 귀무가설을 기각합니다. 실제 매출 평균(약 632,653원)이 600,000원과 통계적으로 유의하게 다르다는 뜻입니다. 반올림 결과가 0.0으로 표시되지만, 이는 정확히 0이라는 뜻이 아니라 유의수준보다 압도적으로 작다는 의미입니다.

**셀 4) 매출액과 각 변수의 피어슨 상관계수 계산**

In [13]:
corrs = df.drop(columns=['id', 'sales']).corrwith(df['sales'])
print(corrs)

visit_count          0.482586
ad_spend             0.744492
email_sent           0.140247
social_engagement    0.308473
dtype: float64


`corrwith()`는 데이터프레임의 각 컬럼과, 괄호 안에 지정한 Series(여기서는 sales) 사이의 피어슨 상관계수를 한 번에 계산해주는 함수입니다. `id`와 `sales` 자기 자신은 비교 대상에서 제외한 뒤 나머지 네 변수에 대해 각각 상관계수를 구했습니다. 만약 이 함수를 모른다면, `df['sales'].corr(df['visit_count'])`처럼 변수 하나하나에 대해 `.corr()`을 반복 호출해도 같은 결과를 얻을 수 있습니다.

**셀 5) 가장 강한 상관계수 찾기 (문제 2-2 정답)**

In [14]:
answer_2_2 = round(corrs.abs().max(), 3)
strongest_var = corrs.abs().idxmax()

print("가장 강한 상관관계 변수:", strongest_var)
print("2-2 정답:", answer_2_2)

가장 강한 상관관계 변수: ad_spend
2-2 정답: 0.744


상관계수는 -1에서 1 사이의 값을 가지며, 양수든 음수든 0에서 멀수록(절댓값이 클수록) 관계가 강하다는 뜻입니다. 문제에서 "절댓값이 가장 큰 값"을 구하라고 했으므로, `.abs()`로 부호를 무시한 크기만 비교했습니다. ad_spend(광고비)가 매출과 가장 강하게 연관되어 있다는 것을 알 수 있습니다.

> 💡 상관계수 자체는 음수일 수도 있습니다. 만약 어떤 변수의 상관계수가 -0.8이고 다른 변수가 0.6이라면, 절댓값 기준으로는 -0.8쪽이 더 강한 관계입니다. `.abs()`를 빠뜨리면 이런 음의 상관관계를 놓칠 수 있으므로 반드시 절댓값으로 비교해야 합니다.

**셀 6) 회귀분석 수행**

In [15]:
import statsmodels.api as sm

X = sm.add_constant(df.drop(columns=['id', 'sales']))
y = df['sales']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.864
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     372.0
Date:                Mon, 20 Jul 2026   Prob (F-statistic):          2.22e-100
Time:                        22:37:44   Log-Likelihood:                -2911.5
No. Observations:                 240   AIC:                             5833.
Df Residuals:                     235   BIC:                             5850.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const              2.781e+05   1.08e+0

네 변수 모두 p-value가 0.05보다 작아 유의하지만, 그 중에서도 p-value의 크기(작은 정도)에는 차이가 있습니다.

**셀 7) 가장 유의한 변수의 회귀계수 (문제 2-3 정답)**

In [16]:
min_pvar = model.pvalues.drop('const').idxmin()
print("가장 유의한 변수:", min_pvar, "/ p-value:", model.pvalues[min_pvar])

answer_2_3 = round(model.params[min_pvar], 3)
print("2-3 정답:", answer_2_3)

가장 유의한 변수: ad_spend / p-value: 5.885042779865857e-80
2-3 정답: 613.82


`model.pvalues`에서 `.drop('const')`로 절편을 제외한 뒤, `idxmin()`으로 p-value가 가장 작은(가장 유의한) 변수의 이름을 찾았습니다. `idxmax()`가 "가장 큰 값의 위치"를 찾는 것과 반대로, `idxmin()`은 "가장 작은 값의 위치"를 찾습니다. p-value는 작을수록 강하게 유의하다는 뜻이므로, 여기서는 최솟값을 찾는 것이 "가장 유의한 변수"를 찾는 것과 같습니다. 흥미롭게도 셀 5에서 상관계수가 가장 강했던 변수(ad_spend)와 회귀분석에서 가장 유의했던 변수가 일치하는데, 이는 우연이 아니라 매출과 가장 밀접하게 움직이는 변수일수록 회귀모형 안에서도 강하게 유의한 경향이 있기 때문입니다.


### 전체 코드

In [17]:
import pandas as pd
import statsmodels.api as sm
from scipy.stats import ttest_1samp

df = pd.read_csv('ad_campaign_sales.csv')

# 2-1: 단일표본 t검정
t_stat, p = ttest_1samp(df['sales'], 600000)
print("2-1 정답:", round(p, 3))

# 2-2: 상관계수 최댓값
corrs = df.drop(columns=['id', 'sales']).corrwith(df['sales'])
print("2-2 정답:", round(corrs.abs().max(), 3))

# 2-3: 회귀분석 후 가장 유의한 변수의 계수
X = sm.add_constant(df.drop(columns=['id', 'sales']))
y = df['sales']
model = sm.OLS(y, X).fit()
min_pvar = model.pvalues.drop('const').idxmin()
print("2-3 정답:", round(model.params[min_pvar], 3))

2-1 정답: 0.0
2-2 정답: 0.744
2-3 정답: 613.82


### 정리

이 문제는 t검정, 상관분석, 회귀분석이라는 세 가지 기법을 한 문제 안에서 순서대로 다뤘습니다. 각 기법이 사용하는 함수(`ttest_1samp`, `corrwith`, `sm.OLS`)는 서로 다르지만, 결국 "통계량 또는 계수를 계산 → p-value나 크기를 비교 → 조건에 맞는 값을 골라낸다"는 흐름은 공통적입니다. 특히 `idxmax()`와 `idxmin()`을 상황에 맞게 구분해서 사용하는 것(상관계수는 절댓값이 클수록, p-value는 값 자체가 작을수록 의미가 있다는 점)이 이 문제의 핵심이었습니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

단일표본 t검정에서 비교하는 기준값(여기서는 600,000원)이 문제마다 다르게 주어질 수 있습니다. 기준값이 바뀌면 t통계량과 p-value도 함께 달라집니다.

**변형 예시: 비교 기준값이 600,000원이 아니라 650,000원이라면**

In [18]:
t_stat2, p2 = ttest_1samp(df['sales'], 650000)
print("t통계량:", t_stat2)
print("p-value:", round(p2, 3))

t통계량: -2.2058553839726582
p-value: 0.028


비교 기준값을 600,000원에서 650,000원으로 바꾸자 t통계량의 부호가 양수(4.15)에서 음수(-2.21)로 바뀌었습니다. 이는 실제 매출 평균(약 632,653원)이 600,000원보다는 크고 650,000원보다는 작기 때문입니다. p-value도 0.028로, 여전히 유의수준 0.05보다는 작아 귀무가설을 기각하지만, 600,000원과 비교했을 때(p≈0.00005)보다는 훨씬 큰 값입니다. 이는 실제 평균이 650,000원에 상대적으로 더 가깝기 때문에, "다르다고 확신하는 정도"가 조금 약해진 것으로 해석할 수 있습니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| 예측 확률과 원본 데이터를 인덱스로 연결 | `pred_prob.idxmax()` 후 `df.loc[인덱스, 컬럼]` | 예측 결과의 위치가 원본 데이터의 같은 위치와 대응됨 (인덱스 정리 필수) |
| 하나의 표본 평균이 특정 값과 같은지 검정 | `ttest_1samp(데이터, 비교값)` | 대응표본 t검정과 달리 비교 대상이 다른 데이터가 아니라 고정된 숫자 |
| 여러 변수와 하나의 변수 간 상관계수를 한 번에 계산 | `df.corrwith(다른 Series)` | 변수마다 `.corr()`을 반복 호출하지 않아도 됨 |
| 상관관계의 강도를 비교할 때 | `.abs()`로 절댓값 비교 | 음의 상관관계도 강한 관계일 수 있음 |
| 가장 유의한(p-value가 가장 작은) 변수 찾기 | `model.pvalues.drop('const').idxmin()` | p-value는 작을수록 강하게 유의하므로 idxmin 사용 |
| 한 문제에서 여러 통계기법을 순서대로 적용 | 기법별로 다른 함수를 사용하되 동일한 데이터를 재활용 | 데이터를 다시 불러올 필요 없이 이미 불러온 데이터프레임을 계속 사용 |